# Voice Chatbot

In [ ]:
from dotenv import load_dotenv
load_dotenv()

1. 사용자 입력을 음성으로 받는다.
2. STT를 적용하여 텍스트로 변환한다.
3. 변환된 텍스트를 입력으로 하여 프롬프트 엔지니어링 -> api 요청
4. 반환받은 응답을 TTS를 적용하여 음성으로 재생한다.

In [5]:
import speech_recognition as sr

recognizer = sr.Recognizer()
speech_prompt = []

while True:
    try:
        with sr.Microphone() as source:
            recognizer.adjust_for_ambient_noise(source, duration=0.5)
            print('말씀하세요. 5초간 입력이 없으면 종료합니다.')
            audio = recognizer.listen(source, timeout=5, phrase_time_limit=30)
            speech = recognizer.recognize_google(audio, language='ko-KR')
            print(speech)
            speech_prompt.append(speech)

        if speech == '종료':
            break

    except sr.WaitTimeoutError:
        print('음성 입력이 없어 종료합니다.')
        break
    
    except sr.UnknownValueError:
        print('이해하지 못했어요. 다시 시도해 주세요.')
        break

    except sr.RequestError:
        print('연결이 좋지 않아요. 다시 시도해 주세요.')
        break

말씀하세요. 5초간 입력이 없으면 종료합니다.
음성 입력이 없어 종료합니다.


In [18]:
speech_prompt = ['안녕하세요. 혹시 화장실은 어디예요?']

In [19]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain.chat_models import init_chat_model
from langchain_core.output_parsers import StrOutputParser

prompt = ChatPromptTemplate.from_messages([
    ('system', '''
# RULE
-ROLE: 중화권 여행 중인 사용자에게 통역의 도움을 주는 Assistant
-GOAL: 중화권(보통화, 광둥어, 중국 사투리 등)의 언어를 통역
# TASK
- {speech_prompt}를 인식 → 상황에 맞는 답변
-기본적으로 한국어를 포함해 응답
## SITUATION
-사용자가 한국어로 말하는 경우: 여행 중 중국인과 소통이 어려움 → 사용자의 요청을 이해하고 사용자가 말하고자 하는 바를 중국어로 출력
-사용자가 중국어로 말하는 경우: 상대방이 사용자와의 소통을 위해 말함 → 사용자를 위해 중국어를 한국어로 번역/전달
# CONSTRAINTS
-각 언어로 번역 시, 원문을 훼손하지 않는 범위에서 의미 중심의 의역을 활용
-자연스러운 구어체 활용
-비속어 인식 시: '나쁜 말이에요. 번역할 수 없어요.' 중국어/한국어 모두 출력
-심한 사투리, 알아들을 수 없는 말 인식 시: '조금 더 천천히 말해 주세요.' 중국어/한국어 모두 출력
'''),
    MessagesPlaceholder(variable_name='history'),
    ('human', '{speech_prompt}')])
llm = init_chat_model('openai:gpt-4.1-mini')
output_parser = StrOutputParser()


In [20]:
from langchain_core.chat_history import BaseChatMessageHistory
from pydantic import BaseModel, Field
from typing import List
from langchain_core.messages import BaseMessage

class InMemoryHistory(BaseChatMessageHistory, BaseModel):
    '''사용자(Session)별 대화내용을 기록하는 클래스'''
    messages: List[BaseMessage] = Field(default_factory=list)

    def add_messages(self, messages: List[BaseMessage]) -> None:
        self.messages.extend(messages)

    def clear(self) -> None:
        self.messages = []

store = {}

In [21]:
def get_by_session_id(session_id: str) -> BaseChatMessageHistory:
    if session_id not in store:
        store[session_id] = InMemoryHistory()
    return store[session_id]

In [22]:
from langchain_core.runnables import RunnableWithMessageHistory

chain = prompt | llm | output_parser

chain_with_memory = RunnableWithMessageHistory(
    chain,
    get_by_session_id,
    input_messages_key='speech_prompt',
    history_messages_key='history'
)

In [24]:
from openai import OpenAI

client = OpenAI()

inputs = chain_with_memory.invoke(
    {'speech_prompt': ' '.join(speech_prompt)},
    config={
        'configurable': {
            'session_id': '100'
        }
    }
)

tts_output = client.audio.speech.create(
    model='tts-1',
    voice='alloy',
    input=inputs
)
tts_output.stream_to_file('tts_output.mp3')

C:\Users\Playdata\AppData\Local\Temp\ipykernel_12000\814513909.py:19: DeprecationWarning: Due to a bug, this method doesn't actually stream the response content, `.with_streaming_response.method()` should be used instead
  tts_output.stream_to_file('tts_output.mp3')


In [26]:
from IPython.display import Audio
print(inputs)
display(Audio('tts_output.mp3', autoplay=True))

한국어: 안녕하세요. 혹시 화장실은 어디예요?  
중국어(광둥어): 你好，请问厕所喺边度呀？(Nei5 hou2, ceng2 man6 ce3 sou3 hai2 bin1 dou6 aa3?)
